# **Importação das bibliotecas**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf

# **Pré-processamento de dados**

In [ ]:
_URL = 'https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip'
path_to_zip = tf.keras.utils.get_file('cats_and_dogs.zip', origin=_URL, extract=True)
PATH = os.path.join(os.path.dirname(path_to_zip), 'cats_and_dogs_filtered')

train_dir = os.path.join(PATH, 'train')
validation_dir = os.path.join(PATH, 'validation')

BATCH_SIZE = 32
IMG_SIZE = (160, 160)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir, shuffle=True, batch_size=BATCH_SIZE, image_size=IMG_SIZE)

68606236/68606236 [==============================] - 0s 0us/step
Found 2000 files belonging to 2 classes.


In [ ]:
validation_dataset = tf.keras.utils.image_dataset_from_directory(validation_dir,
                                                                 shuffle=True,
                                                                 batch_size=BATCH_SIZE,
                                                                 image_size=IMG_SIZE)

Found 1000 files belonging to 2 classes.


In [ ]:
class_names = train_dataset.class_names
class_names

['cats', 'dogs']

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_dataset.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [ ]:
# Criando um conjunto de dados de teste
val_batches = tf.data.experimental.cardinality(validation_dataset)
test_dataset = validation_dataset.take(val_batches // 5)
validation_dataset = validation_dataset.skip(val_batches // 5)

# **Configurar o conjunto de dados para alto desempenho**

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

# **Aumento de dados / Data Augumentation**

In [ ]:
"""
Quando você não tem um grande conjunto de dados de imagem,
é uma boa prática introduzir artificialmente a diversidade de amostra aplicando
transformações aleatórias, porém realistas, às imagens de treinamento,
como rotação e inversão horizontal. Isso ajuda a expor o modelo a diferentes
aspectos dos dados de treinamento e reduzir o overfitting
"""

data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip('horizontal'),
  tf.keras.layers.RandomRotation(0.2),
])

In [ ]:
for image, _ in train_dataset.take(1):
  plt.figure(figsize=(10, 10))
  first_image = image[0]
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    augmented_image = data_augmentation(tf.expand_dims(first_image, 0))
    plt.imshow(augmented_image[0] / 255)
    plt.axis('off')

In [ ]:
"""
o modelbase espera valores de pixel em [-1, 1] ,
mas os valores de pixel estão em [0, 255]
tf.keras.applications.MobileNetV2 corrigira isso.
"""
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
"""Alternativa 2"""
rescale = tf.keras.layers.Rescaling(1./127.5, offset=-1)

# **Modelo Base**

In [ ]:
# Crie o modelo base a partir do modelo pré-treinado MobileNet V2
IMG_SHAPE = IMG_SIZE + (3,)
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SHAPE,
                                               include_top=False,
                                               weights='imagenet')

9406464/9406464 [==============================] - 0s 0us/step


In [ ]:
image_batch, label_batch = next(iter(train_dataset))
feature_batch = base_model(image_batch)
print(feature_batch.shape)

(32, 5, 5, 1280)


In [ ]:
base_model.trainable = False

In [ ]:
# Vamos ver a arquitetura do modelo base
base_model.summary()

## **Cabeçalho**

In [ ]:
# redução da dimensionalidade
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
feature_batch_average = global_average_layer(feature_batch)
print(feature_batch_average.shape)

(32, 1280)


In [ ]:
# camada de saída
prediction_layer = tf.keras.layers.Dense(1)
prediction_batch = prediction_layer(feature_batch_average)
print(prediction_batch.shape)

(32, 1)


# **Definição do modelo**

In [ ]:
inputs = tf.keras.Input(shape=(160, 160, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = global_average_layer(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = prediction_layer(x)
model = tf.keras.Model(inputs, outputs)

In [ ]:
base_learning_rate = 0.0001
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=base_learning_rate),
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 160, 160, 3)]     0         
                                                                 
 sequential (Sequential)     (None, 160, 160, 3)       0         
                                                                 
 tf.math.truediv (TFOpLambd  (None, 160, 160, 3)       0         
 a)                                                              
                                                                 
 tf.math.subtract (TFOpLamb  (None, 160, 160, 3)       0         
 da)                                                             
                                                                 
 mobilenetv2_1.00_160 (Func  (None, 5, 5, 1280)        2257984   
 tional)                                                         
                                                             

In [ ]:
len(model.trainable_variables)

2

In [ ]:
initial_epochs = 10

loss0, accuracy0 = model.evaluate(validation_dataset)

26/26 [==============================] - 17s 554ms/step - loss: 0.8809 - accuracy: 0.4542


In [ ]:
print("initial loss: {:.2f}".format(loss0))
print("initial accuracy: {:.2f}".format(accuracy0))

initial loss: 0.88
initial accuracy: 0.45


# **Treinamento do modelo**

In [ ]:
history = model.fit(train_dataset,
                    epochs=initial_epochs,
                    validation_data=validation_dataset)

Epoch 1/10
63/63 [==============================] - 66s 979ms/step - loss: 0.8189 - accuracy: 0.5030 - val_loss: 0.6027 - val_accuracy: 0.6510
Epoch 2/10
63/63 [==============================] - 61s 964ms/step - loss: 0.5871 - accuracy: 0.6765 - val_loss: 0.4343 - val_accuracy: 0.7797
Epoch 3/10
63/63 [==============================] - 61s 960ms/step - loss: 0.4738 - accuracy: 0.7465 - val_loss: 0.3270 - val_accuracy: 0.8639
Epoch 4/10
63/63 [==============================] - 53s 834ms/step - loss: 0.3852 - accuracy: 0.8075 - val_loss: 0.2699 - val_accuracy: 0.8861
Epoch 5/10
63/63 [==============================] - 63s 994ms/step - loss: 0.3398 - accuracy: 0.8460 - val_loss: 0.2257 - val_accuracy: 0.9146
Epoch 6/10
63/63 [==============================] - 65s 1s/step - loss: 0.3028 - accuracy: 0.8660 - val_loss: 0.1951 - val_accuracy: 0.9270
Epoch 7/10
63/63 [==============================] - 67s 1s/step - loss: 0.2635 - accuracy: 0.8890 - val_loss: 0.1740 - val_accuracy: 0.9431
Epoch

## **Curvas de aprendizado**

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([min(plt.ylim()),1])
plt.title('Training and Validation Accuracy')

plt.subplot(2, 1, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0,1.0])
plt.title('Training and Validation Loss')
plt.xlabel('epoch')
plt.show()

# **Afinação do modelo**

In [ ]:
base_model.trainable = True

In [ ]:
# Vamos dar uma olhada para ver quantas camadas existem no modelo base
print("Number of layers in the base model: ", len(base_model.layers))

# Ajuste a partir desta camada
fine_tune_at = 100

# Congele todas as camadas antes da camada `fine_tune_at {100}`
for layer in base_model.layers[:fine_tune_at]:
  layer.trainable = False

Number of layers in the base model:  154


## **Compilar o modelo**

In [ ]:
model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              optimizer = tf.keras.optimizers.RMSprop(learning_rate=base_learning_rate/10),
              metrics=['accuracy'])

model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 160, 160, 3)]     0         
                                                                 
 sequential (Sequential)     (None, 160, 160, 3)       0         
                                                                 
 tf.math.truediv (TFOpLambd  (None, 160, 160, 3)       0         
 a)                                                              
                                                                 
 tf.math.subtract (TFOpLamb  (None, 160, 160, 3)       0         
 da)                                                             
                                                                 
 mobilenetv2_1.00_160 (Func  (None, 5, 5, 1280)        2257984   
 tional)                                                         
                                                             

In [ ]:
len(model.trainable_variables)

56

### **Continuar treinamento**

In [ ]:
fine_tune_epochs = 10
total_epochs =  initial_epochs + fine_tune_epochs

history_fine = model.fit(train_dataset,
                         epochs=total_epochs,
                         initial_epoch=history.epoch[-1],
                         validation_data=validation_dataset)

Epoch 10/20
63/63 [==============================] - 90s 1s/step - loss: 0.1520 - accuracy: 0.9350 - val_loss: 0.0561 - val_accuracy: 0.9827
Epoch 11/20
63/63 [==============================] - 91s 1s/step - loss: 0.1162 - accuracy: 0.9530 - val_loss: 0.0556 - val_accuracy: 0.9814
Epoch 12/20
63/63 [==============================] - 83s 1s/step - loss: 0.0990 - accuracy: 0.9575 - val_loss: 0.0658 - val_accuracy: 0.9728
Epoch 13/20
63/63 [==============================] - 82s 1s/step - loss: 0.0855 - accuracy: 0.9605 - val_loss: 0.0449 - val_accuracy: 0.9814
Epoch 14/20
63/63 [==============================] - 81s 1s/step - loss: 0.0944 - accuracy: 0.9670 - val_loss: 0.0554 - val_accuracy: 0.9777
Epoch 15/20
63/63 [==============================] - 85s 1s/step - loss: 0.0862 - accuracy: 0.9615 - val_loss: 0.0414 - val_accuracy: 0.9839
Epoch 16/20
63/63 [==============================] - 86s 1s/step - loss: 0.0720 - accuracy: 0.9725 - val_loss: 0.0371 - val_accuracy: 0.9864
Epoch 17/20
6

In [ ]:
acc += history_fine.history['accuracy']
val_acc += history_fine.history['val_accuracy']

loss += history_fine.history['loss']
val_loss += history_fine.history['val_loss']

In [ ]:
plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.ylim([0.8, 1])
plt.plot([initial_epochs-1,initial_epochs-1],
          plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(2, 1, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.ylim([0, 1.0])
plt.plot([initial_epochs-1,initial_epochs-1],
         plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('epoch')
plt.show()

In [ ]:
loss, accuracy = model.evaluate(test_dataset)
print('Test accuracy :', accuracy)

6/6 [==============================] - 4s 554ms/step - loss: 0.0447 - accuracy: 0.9896
Test accuracy : 0.9895833134651184


In [ ]:
# Recuperar um lote de imagens do conjunto de testes
image_batch, label_batch = test_dataset.as_numpy_iterator().next()
predictions = model.predict_on_batch(image_batch).flatten()

# Aplique um sigmóide, pois nosso modelo retorna logits
predictions = tf.nn.sigmoid(predictions)
predictions = tf.where(predictions < 0.5, 0, 1)

print('Predições:\n', predictions.numpy())
print('Labels:\n', label_batch)

plt.figure(figsize=(10, 10))
for i in range(9):
  ax = plt.subplot(3, 3, i + 1)
  plt.imshow(image_batch[i].astype("uint8"))
  plt.title(class_names[predictions[i]])
  plt.axis("off")